# More on Consuming Messages

Kafi uses a lot of default configurations for making consuming messages as effortless as possible.

There are two ways of consuming messages:
1. Using your own consume/event loop.
2. Using Kafi's built-in consume/event loop.


## Using Your Own Consume/Event Loop

Imagine you set up your code with your own consume/event loop:
```python
co = c.consumer("topic_json")
while True:
    m_list = co.consume()
    # do something with the messages
    co.commit()
```
What is happening under the covers here? All the configuration parameters are taken from the `kafi` section of the corresponding storage configuration file, e.g. `configs/clusters/local.yaml`. You can also reconfigure at runtime by using the corresponding configuration keys with underscores instead of full stops (and Python types, e.g. strings or bools for the values), e.g. `c.enable_auto_commit(True)` at runtime corresponds to `enable.auto.commit: true` in the configuration.

* Kafi implicitly creates a new consumer group name (=the current timestamp in ms). If you have set `consumer.group.prefix` in the `kafi` section of the storage configuration file, the consumer group name becomes `<consumer.group.prefix><current timestamp in ms>`. You can set your own consumer group using the `group` parameter, e.g. `co = c.consumer("test-topic", group=<your own consumer group name>)`.
* Kafi uses the configured `auto.offset.reset` value (default: `earliest`).
* Kafi uses the configured `enable.auto.commit` value (default: `false`).
* Kafi uses the configured `consume.batch.size` value (default: `100`)
* The default key type is set to the configured `key.type` value (default: `str`)
* The default value type is set to the configured `value.type` value (default: `json`)
* The following configurations are only supported for the `Cluster` storage, i.e., native Kafka:
    * Kafi uses the configured `session.timeout.ms` value (default: `45000`)
    * Kafi uses the configured `isolation.level` value (default: `read_uncommitted`)
    * Kafi uses the configured `cluster.consume.timeout` value (default: `0.1`) - this is the consume timeout for individual calls to `consume()` using the Confluent Python client.
* Kafi sets the maximum number of messages to consume to `ALL_MESSAGES` (=`-1`)

The consume/event loop above could also be spelled out (=the defaults spelled out) as follows:
```python
import time
ALL_MESSAGES=-1
#
c.auto_offset_reset("earliest") # config file: auto.offset.reset: earliest
c.consume_batch_size(100) # config file: consume.batch.size: 100
c.enable_auto_commit(False) # config file: enable.auto.commit: false
c.session_timeout_ms(45000) # config file: session.timeout.ms: 45000
c.isolation_level("read_uncommitted") # config file: isolation.level: read_uncommitted
c.cluster_consume_timeout(0.1) # config file: cluster.consume.timeout: 0.1
c.key_type("str") # config file: key.type: str
c.value_type("json") # config file: value.type: json
#
co = c.consumer("topic_json", group=f"{c.consumer_group_prefix()}{time.time() * 1000}")
# co = c.consumer("topic_json", group=f"{c.consumer_group_prefix()}{time.time() * 1000}", key_type="str", value_type="json") - if you'd like to set key_type and value_type here
while True:
    co.consume(n=ALL_MESSAGES)
    # do something with the messages
    co.commit()
```

## Using Kafi's built-in consume/event loop.

All the convenient "functional" and "shell" methods of Kafi use Kafi's *built-in consume/event loop*, e.g. `foreach`, `map`, `cat`, `grep`...

The built-in consume/event loop adds more configuration items:
* `commit.after.processing`: If set to true, commit all processed messages. Do not commit otherwise. This setting is *independent* of the `enable.auto.commit` setting - if you set `commit.after.processing` to true, you must make sure to set `enable.auto.commit` to false (Kafi default).
* `consume.timeout`: After no more messages could be consumed for the time specified in `consume.timeout`, Kafi's built-in consume/event loop stops.
